**EVALUATION OF YOLO_MEDIUM ON TEST SET**
- mainly for debuggin / testing purposes
- complete and standardized evaluation script is 'src/evaluation/yolo_eval.py'

In [1]:
from pathlib import Path
import os
from tqdm import tqdm
from ultralytics import YOLO
import numpy as np

In [2]:
IMSIZE = 640
MODEL = 'best_v11n_640_default_1000e.pt'
CONF=0.5
IOU=0.5

# ============== TEST SET ============== #

TEST_IMAGES_PATH = Path(f'../data/processed_{IMSIZE}/images/test')
TEST_LABELS_PATH = Path(f'../data/processed_{IMSIZE}/labels/test')
TEST_SET_YAML_PATH = Path(f'../data/YOLO_CONFIGS/test_{IMSIZE}.yaml')

# ============== OUR DATA ============== #
NO_STAMPS_PATH = Path(f'../data/raw/unlabeled/dataset/bg')
STAMPED_PATH = Path(f'../data/raw/unlabeled/dataset/stamps')

# ==============  MODEL  ============== #
MODELS_PATH = Path('../src/trained_models')

model = YOLO(MODELS_PATH / MODEL)

**TEST SET EVALUATION**

In [ ]:
### number of bounding boxes evaluation ###

images = [f for f in os.listdir(TEST_IMAGES_PATH) if f.endswith(('.jpg', '.jpeg', '.png'))]

count_abs_errors = [] # predicted stamps - actual stamps
exact_matches = 0 # detected number of stamps == actual number of stamps

for img_name in tqdm(images):
    img_path = os.path.join(TEST_IMAGES_PATH, img_name)
    label_path = os.path.join(TEST_LABELS_PATH, os.path.splitext(img_name)[0] + '.txt')

    with open(label_path) as f:
        stamp_count = len([line for line in f if line.strip()])
    
    results = model(img_path, conf=CONF, iou=IOU, verbose=False)[0]
    pred_count = len(results.boxes)
    count_abs_errors.append(abs(pred_count - stamp_count))

    if pred_count == stamp_count:
        exact_matches += 1

mae = np.mean(count_abs_errors)
exact_pct = exact_matches / len(images) * 100

print("\n=== Counting Performance ===")
print(f"Mean Absolute Error (count)   : {mae:.4f}")
print(f"Exact count correct        : {exact_pct:.1f}% ({exact_matches}/{len(images)}) images)")

100%|██████████| 16/16 [00:00<00:00, 28.27it/s]


=== Counting Performance ===
Mean Absolute Error (count)   : 0.0000
Exact count correct        : 100.0% (16/16) images)
Average |pred − gt|         : 0.00 stamps per image


In [4]:
metrics = model.val(
    data       = TEST_SET_YAML_PATH,
    conf       = CONF,
    iou        = IOU,
    cls        = 0.001,      # minimize effectof classification metric 
    plots      = True,
    save_json  = True,
    project    = "test_evaluation",
    name       = MODEL,
    exist_ok   = True,
    verbose    = True
)

print(f"Box mAP@0.5     : {metrics.box.map50:.4f}")      # main number for position accuracy
print(f"Box mAP@0.5:0.95: {metrics.box.map:.4f}")

Ultralytics 8.3.228 🚀 Python-3.11.7 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3080, 9902MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3549.3±1231.7 MB/s, size: 95.0 KB)
val: Scanning /home/diaz_asian/StampDetector/data/processed_640/labels/test.cache... 16 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 16/16 44.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.2it/s 0.2s
                   all         16         12          1          1      0.995      0.887
Speed: 1.2ms preprocess, 3.6ms inference, 0.0ms loss, 0.3ms postprocess per image
Saving /home/diaz_asian/StampDetector/notebooks/test_evaluation/best_v11n_640_default_1000e.pt/predictions.json...
Results saved to /home/diaz_asian/StampDetector/notebooks/test_evaluation/best_v11n_640_default_1000e.pt
Box mAP@0.5     : 0.9950
Box mAP@0.5:0.95: 0.8872


**OUR DATA EVALUATION**

In [5]:
from preprocessing import yolop
import cv2

In [ ]:
### performance on stamp-free dataset ###

images = os.listdir(NO_STAMPS_PATH)

for img_file in images:
    img = cv2.imread(str(NO_STAMPS_PATH / img_file))

    img_processed, _, _ = yolop.resize_and_pad(img,IMSIZE)
    results = model(img_processed, conf=CONF, iou=IOU, verbose=False)[0]
    pred_count = len(results.boxes)
    print(pred_count)

# 3 / 9 --> wrong

0
0
0
0
1
0
2
3
0


In [ ]:
### perfomance on stamped dataset ###

images = os.listdir(STAMPED_PATH)
print(images)

for img_file in images:
    img = cv2.imread(str(STAMPED_PATH / img_file))

    img_processed, _, _ = yolop.resize_and_pad(img,IMSIZE)
    results = model(img_processed, conf=CONF, iou=IOU, verbose=False)[0]
    pred_count = len(results.boxes)
    print(pred_count)

# 10 / 12 detected (how correctly is anoher question of course :D)

['faktúra_pdf0.jpg', 'faktúra_pdf1.jpg', 'Faktura_2016275_12_2016_pdf1.jpg', 'kostýmyMŠ_ZŠ_8.2.2017_zálohováfaktura_2PDF.jpg', 'faktura_237279pdf.jpg', '38842855-Factura-Metro.png', 'Faktura_2016275_12_2016_pdf0.jpg', 'Faktura_2016275_12_2016_pdf10.jpg', 'f_2011_094.pdf.jpg', 'Faktura_2016275_12_2016_pdf2.jpg', 'Papercon_13_40012_colorstav2130553_pdf.jpg', 'faktúra_pdf2.jpg']
1
1
1
1
1
0
4
0
2
1
3
2
